<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=312712246" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import os
from collections import Counter, deque

is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

# ================================
# LOAD DATA
# ================================
if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(path) as f:
    data = json.load(f)

# ================================
# BASIC UTIL
# ================================
def copy_grid(g):
    return [row[:] for row in g]

def zero_grid(h, w):
    return [[0]*w for _ in range(h)]

def most_common_color(g):
    return Counter([c for r in g for c in r]).most_common(1)[0][0]

def fill_color(h, w, c):
    return [[c]*w for _ in range(h)]

# ================================
# TRANSFORMATIONS
# ================================
def rotate90(g):
    return list(zip(*g[::-1]))

def rotate180(g):
    return rotate90(rotate90(g))

def rotate270(g):
    return rotate90(rotate180(g))

def flip_h(g):
    return [row[::-1] for row in g]

def flip_v(g):
    return g[::-1]

def to_list(g):
    return [list(row) for row in g]

# ================================
# SIMILARITY
# ================================
def similarity(a, b):
    if len(a) != len(b) or len(a[0]) != len(b[0]):
        return 0
    same = 0
    total = len(a)*len(a[0])
    for i in range(len(a)):
        for j in range(len(a[0])):
            if a[i][j] == b[i][j]:
                same += 1
    return same / total

# ================================
# BFS OBJECT DETECTION
# ================================
def find_objects(grid):
    h, w = len(grid), len(grid[0])
    visited = [[False]*w for _ in range(h)]
    objects = []

    for i in range(h):
        for j in range(w):
            if grid[i][j] != 0 and not visited[i][j]:
                color = grid[i][j]
                q = deque([(i, j)])
                visited[i][j] = True
                pixels = []

                while q:
                    x, y = q.popleft()
                    pixels.append((x, y))

                    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nx, ny = x+dx, y+dy
                        if 0 <= nx < h and 0 <= ny < w:
                            if not visited[nx][ny] and grid[nx][ny] == color:
                                visited[nx][ny] = True
                                q.append((nx, ny))

                objects.append(pixels)

    return objects

# ================================
# SHIFT DETECTION
# ================================
def find_shift(train):
    shifts = []
    for pair in train:
        inp, out = pair["input"], pair["output"]
        obj_in = find_objects(inp)
        obj_out = find_objects(out)

        if len(obj_in) == 1 and len(obj_out) == 1:
            x1, y1 = obj_in[0][0]
            x2, y2 = obj_out[0][0]
            shifts.append((x2-x1, y2-y1))

    if shifts:
        return Counter(shifts).most_common(1)[0][0]
    return (0, 0)

def apply_shift(grid, shift):
    h, w = len(grid), len(grid[0])
    dx, dy = shift
    new = zero_grid(h, w)

    for i in range(h):
        for j in range(w):
            if grid[i][j] != 0:
                ni, nj = i+dx, j+dy
                if 0 <= ni < h and 0 <= nj < w:
                    new[ni][nj] = grid[i][j]

    return new

# ================================
# GENERATE CANDIDATES
# ================================
def generate_candidates(inp, shift):
    h, w = len(inp), len(inp[0])
    cands = []

    cands.append(copy_grid(inp))
    cands.append(zero_grid(h, w))
    cands.append(fill_color(h, w, most_common_color(inp)))
    cands.append(apply_shift(inp, shift))

    # rotations
    cands.append(to_list(rotate90(inp)))
    cands.append(to_list(rotate180(inp)))
    cands.append(to_list(rotate270(inp)))

    # flips
    cands.append(flip_h(inp))
    cands.append(flip_v(inp))

    return cands

# ================================
# MAIN SOLVER
# ================================
def solve_task(task):
    results = []

    shift = find_shift(task["train"])

    # ambil target pattern dari training terakhir
    target = task["train"][-1]["output"] if task["train"] else None

    for test_case in task["test"]:
        inp = test_case["input"]

        candidates = generate_candidates(inp, shift)

        scored = []
        for c in candidates:
            if target:
                score = similarity(c, target)
            else:
                score = 0
            scored.append((score, c))

        scored.sort(reverse=True, key=lambda x: x[0])

        best1 = scored[0][1]
        best2 = scored[1][1] if len(scored) > 1 else best1

        results.append({
            "attempt_1": best1,
            "attempt_2": best2
        })

    return results

# ================================
# BUILD SUBMISSION
# ================================
submission = {}

for task_id, task in data.items():
    submission[task_id] = solve_task(task)

# ================================
# SAVE
# ================================
with open("submission.json", "w") as f:
    json.dump(submission, f)

print("V23 Pattern+Rotation submission READY!")

V23 Pattern+Rotation submission READY!
